# 방사청 국내조달·업체 정제 → `clean_` 7표 적재

규칙 근거는 `docs/reference/data-cleaning-rules.md` §2-2·§2-9·§2-10·§3, DDL은 `db/schema.sql`, 조인 키 실측은 `db/query_join_key_check.sql`.

| 원본 파일(데이터셋 키) | 정제 결과(clean_) | 기대 행 수 | 비고 |
|---|---|---|---|
| `raw_dapa_contract` 43,112 | `clean_dapa_contract` — 계약번호×차수(0→00), 최종 차수 `is_latest_seq` | 43,105 (+ `clean_excluded_row` KEY_CONFLICT 1 · PLACEHOLDER 6 테스트 업체) | 5분류·속성은 기본값(판단 보류/미확인) |
| `raw_dapa_bid_notice` 10,842 | `clean_dapa_bid_notice` — 참조공고번호+차수 | 10,840 (+ COL_SHIFT 2) | |
| `raw_dapa_bid_result` 7,405 | `clean_dapa_bid_result` — 공고번호+차수+`result_seq`, `dup_kind` 3분류 | 7,403 (+ COL_SHIFT 2) | 공고와 행 조인 없음, `notice_link_status`만 |
| `raw_dapa_domestic_plan` 35,859 | `clean_dapa_domestic_plan` — raw 1:1 | 35,859 | `budget_krw` NULL = 미기재 4,965 |
| `raw_dapa_contract_exec_by_service` 40 | `clean_dapa_contract_exec_by_service` | 40 | |
| `raw_dapa_defense_company` 84 | `clean_dapa_defense_company` — 지정 업체 1:1(`v_defense_company_sector` 원천) | 84 | |
| 계약 + 낙찰 사업자번호 | `clean_company` — 사업자번호 고유 | 14,836 (테스트 업체 2 제외) | `name_norm` 잠정 규칙 |
| 방산업체 84 + B2 계약업체 407 | `clean_company_name_link` — 이름 매칭 결과 | 84 + 407 | exact / multi / none |

원칙
- 원본은 `data/raw/` 파일을 `read_raw()`로 읽는다(RDS에는 `raw_` 표를 두지 않는다 — `raw_row_id` = 파서 순번, 파일명 정렬 × 행 순). 대상 `clean_`이 **비어 있을 때만** 적재한다(재적재는 `TRUNCATE` 후 재실행). 제외한 raw 행은 전부 `clean_excluded_row`에 사유와 함께 남겨 `raw = clean + excluded`를 검산한다.
- **판단 속성은 만들지 않는다**: `class5`=`판단 보류`, `is_electronic`/`is_part`/`is_defense_related`/`is_target_b1`/`is_completed_b2`/`domestic_mfg_status`=`미확인`, `review_status`=`후보`(품목군 연결 열 `contract_group`·`category`·`category_link_status`는 두지 않는다 — 카테고리 맵을 쓰지 않는다). 키워드 규칙·검수는 별도 단계(`schema-design.md` §7-5).
- 값 추정 금지: 주소 첫 토큰이 `ref_sido_map`(44 토큰)에 없으면 `sido_code` NULL(계약 2행·2토큰: `**`·`1` — 용산구 2행은 테스트 업체라 제외됨). `광주`·`광주시`는 표에 없고 둘째 토큰으로 29/41 판별, 판별 불가는 NULL. 예산 미기재는 NULL(0 아님).
- 대표 행은 `row_id` 최소. 충돌·중복은 지우지 않고 `seq_conflict_flag`·`conflict_raw_row_ids`·`result_seq`·`dup_kind`로 보존한다.
- `name_norm` 잠정 규칙: `(주)`·`㈜`·`주식회사`·`(유)`·`유한회사` 제거 후 공백 제거 — `clean_dapa_localized_item.contractor_name_norm`과 같은 규칙.
- 단계별 건수는 `meta_load_log`에 기록한다. 접속은 `scripts/load_db.py`의 `connect()`(→ `.env`, `etl_rw`).

In [1]:
import sys, re
from pathlib import Path
from datetime import datetime
import pandas as pd
import numpy as np

ROOT = Path.cwd() if (Path.cwd() / "scripts").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "scripts"))
from load_db import connect, log_stage, current_user, read_raw, log_raw_stage   # .env 읽기 · meta_load_log 기록 · 원본 파일 읽기
RAW = {}   # 원본 파일 DataFrame 캐시 {데이터셋 키: read_raw 결과}. RDS 에는 raw_ 표가 없다 — row_id = 파서 순번 = clean.raw_row_id
def raw(table, cols=None):
    if table not in RAW:
        RAW[table] = read_raw(table)
    df = RAW[table]
    return (df[["row_id"] + [c for c in cols if c != "row_id"]] if cols else df).copy()

NB_TAG = "notebooks/04_clean_domestic.ipynb"
conn = connect()
cur = conn.cursor()
USER = current_user(cur)
NOW = datetime.now().replace(microsecond=0)
def q(sql, params=None):
    cur.execute(sql, params)
    return pd.DataFrame(cur.fetchall(), columns=[d[0] for d in cur.description])
def count(t):
    cur.execute(f"SELECT COUNT(*) FROM {t}"); return cur.fetchone()[0]
def py(v):
    if v is None: return None
    if isinstance(v, float) and np.isnan(v): return None
    if v is pd.NaT or (isinstance(v, pd._libs.missing.NAType)): return None
    if hasattr(v, "item") and not isinstance(v, str): return v.item()
    if isinstance(v, pd.Timestamp): return v.to_pydatetime().date()
    return v
def insert_df(table, df, cols, chunk=2000):
    rows = [tuple(py(v) for v in r) for r in df[cols].itertuples(index=False, name=None)]
    sql = f"INSERT INTO {table} ({','.join(cols)}) VALUES ({','.join(['%s']*len(cols))})"
    for i in range(0, len(rows), chunk):
        cur.executemany(sql, rows[i:i+chunk])
    return len(rows)

DATE_RE = r"^\d{4}-\d{2}-\d{2}$"
def to_date(s):
    return pd.to_datetime(s.where(s.astype(str).str.fullmatch(DATE_RE), None), errors="coerce").dt.date
def to_int(s):
    # 숫자 문자열 → 정수(Int64). 소수·지수 표기는 float 경유 후 반올림, 그 외는 NULL
    x = pd.to_numeric(s.replace({"": None}), errors="coerce")
    return x.round().astype("Int64")
def to_dec(s):
    return pd.to_numeric(s.replace({"": None}), errors="coerce")
def yn(s):
    return s.map({"Y": 1, "N": 0}).astype("Int64")
LEGAL = ["(주)", "㈜", "주식회사", "(유)", "유한회사"]
def name_norm(s):
    out = s.fillna("").astype(str).str.strip()
    for t in LEGAL:
        out = out.str.replace(t, "", regex=False)
    return out.str.replace(r"\s+", "", regex=True)
SIDO = dict(q("SELECT token, sido_code FROM ref_sido_map").itertuples(index=False, name=None))
def sido_code(addr):
    # 첫 토큰 → ref_sido_map(44 토큰). '광주'·'광주시'는 표에 없으므로
    # 둘째 토큰으로 광주광역시(29: 동·서·남·북·광산구) / 경기 광주시(41: 읍·면) 판별, 그 밖의 광주는 NULL(AMBIGUOUS_GWANGJU)
    tok = addr.fillna("").astype(str).str.strip().str.split()
    out = tok.str[0].map(SIDO)
    gj = tok.str[0].isin(["광주", "광주시"])
    out[gj & tok.str[1].isin(["동구", "서구", "남구", "북구", "광산구"])] = "29"
    out[gj & tok.str[1].fillna("").str.contains(r"[읍면]$")] = "41"
    return out
excluded = []   # (table_name, raw_row_id, reason_code, note)
print("connected as", USER, "| pandas", pd.__version__, "| ref_sido_map tokens", len(SIDO))

connected as etl_rw@% | pandas 3.0.5 | ref_sido_map tokens 44


## 1. 계약정보 → `clean_dapa_contract`

- 키 = `contract_no` × `contract_seq_norm`(`LPAD(seq,2,'0')`, raw `'0'` 837·`'00'` 34,365 혼재). 정규화 후 고유 43,111, 충돌 1키(`2024UMM1504`-`01` 2행: 27열 동일 — 대표 행 `row_id` 최소, 나머지는 `clean_excluded_row` KEY_CONFLICT + `conflict_raw_row_ids`).
- `is_latest_seq` = 계약번호별 최대 차수(문자열 2자리 비교 = 정수 비교, 차수 ≤ 08) → 37,602행(테스트 업체 6행 제외 후, 계약번호 수와 같음).
- `계약기간` `YYYY-MM-DD~YYYY-MM-DD` 단일 패턴(100%) → `period_start`/`period_end`. 종료일이 2035년 이후면 `period_anomaly_flag=1`(실측 1행, 2525-01-16).
- 금액: `contract_amount`·`total_contract_amount`(NULL 1행)·`reserve_price`(소수 677행 반올림) → BIGINT. `joint_contract_yn` = `공동계약`→1.
- 담당자명 2열·대표자명은 읽지 않는다. `vendor_address`는 `sido_code` 파생용으로만 넣고 화면 미노출.
- **테스트 업체 6행 제외**: 대표업체명이 정확히 `조달테스트업체Ⅰ`(2행, 주소도 `용산구 … 테스트`)·`테스트업체1`(4행)인 행 → `clean_excluded_row` PLACEHOLDER. 키워드 '테스트'로 거르지 않는다(이테스트 주식회사·(주)테스트링크·결함테스트기·콘테스트 등 실제 업체·품명 8행). `is_latest_seq`는 제외 뒤 계산.
- `sido_code`: 첫 토큰 → `ref_sido_map`(44 토큰 — `충남대전시`→30 포함, `광주`·`광주시`는 없음). `광주`·`광주시`는 둘째 토큰으로 29(동·서·남·북·광산구)/41(읍·면) 판별. 미매핑(`**`·`1`)·판별 불가는 NULL 2행(용산구 2행은 테스트 업체로 제외).

In [2]:
rc = raw("raw_dapa_contract", ["contract_no", "contract_seq", "contract_name", "biz_type_name", "contract_form_name", "contract_method_name", "joint_contract_yn", "contract_date", "contract_period", "contract_amount", "total_contract_amount", "reserve_price", "contract_org_name", "demand_org_name", "vendor_name", "vendor_biz_reg_no", "vendor_address", "contract_type"])
print("원본 전체:", len(rc))
rc["contract_seq_norm"] = rc["contract_seq"].astype(str).str.zfill(2)
key = ["contract_no", "contract_seq_norm"]
g = rc.groupby(key, sort=False)
rc["key_n"] = g["row_id"].transform("size")
rc["key_first"] = g["row_id"].transform("min")
conflict = rc[rc["key_n"] > 1].copy()
print("정규화 후 키 고유:", rc[key].drop_duplicates().shape[0], "| 충돌 키:", conflict[key].drop_duplicates().shape[0], "행:", len(conflict))
raw_cols = [c for c in rc.columns if c not in ("row_id", "key_n", "key_first", "contract_seq_norm")]
for (no, sq), grp in conflict.groupby(key):
    diff = [c for c in raw_cols if grp[c].nunique(dropna=False) > 1]
    print(f"  {no}-{sq}: row_id {grp['row_id'].tolist()} | 다른 열: {diff or '없음(원본 완전 중복)'}")
    for rid in grp["row_id"].tolist()[1:]:
        excluded.append(("raw_dapa_contract", int(rid), "KEY_CONFLICT",
                         f"{no}-{sq} 키 중복. 대표 행 row_id {int(grp['row_id'].min())} 채택. 다른 열: {','.join(diff) or '없음(완전 중복)'}"))
c = rc[rc["row_id"] == rc["key_first"]].copy()
c["seq_conflict_flag"] = (c["key_n"] > 1).astype(int)
c["conflict_raw_row_ids"] = None
c["evidence"] = None
for (no, sq), grp in conflict.groupby(key):
    others = ",".join(str(int(r)) for r in grp["row_id"].tolist()[1:])
    diff = [col for col in raw_cols if grp[col].nunique(dropna=False) > 1]
    m = (c["contract_no"] == no) & (c["contract_seq_norm"] == sq)
    c.loc[m, "conflict_raw_row_ids"] = others
    c.loc[m, "evidence"] = f"키 충돌 원본 {grp['row_id'].nunique()}행, 다른 열: {','.join(diff) or '없음(완전 중복)'}. 대표 행 = row_id 최소"
# 테스트 업체 제외: 대표업체명 **정확 일치**만. '테스트'가 든 실제 업체·품명 8행
# (이테스트 주식회사·(주)테스트링크·결함테스트기·콘테스트 …)은 키워드로 걸러내지 않는다. is_latest_seq 계산 전에 뺀다
TEST_VENDORS = ["조달테스트업체Ⅰ", "테스트업체1"]
tv = c["vendor_name"].isin(TEST_VENDORS)
for r in c[tv].itertuples(index=False):
    excluded.append(("raw_dapa_contract", int(r.row_id), "PLACEHOLDER", f"테스트 업체명: {r.vendor_name} ({r.contract_no}-{r.contract_seq_norm})"))
print("테스트 업체 제외:", int(tv.sum()), "행", c.loc[tv, "contract_no"].tolist())   # 기대 6
c = c[~tv].copy()
c["is_latest_seq"] = (c["contract_seq_norm"] == c.groupby("contract_no")["contract_seq_norm"].transform("max")).astype(int)
assert c.groupby("contract_no")["is_latest_seq"].sum().eq(1).all(), "계약번호당 최종 차수 1행 위반"
c["contract_date"] = to_date(c["contract_date"])
assert c["contract_date"].notna().all()
per = c["contract_period"].astype(str).str.split("~", n=1, expand=True)
c["period_start"] = to_date(per[0]); c["period_end"] = to_date(per[1])
c["period_anomaly_flag"] = (pd.to_datetime(c["period_end"]).dt.year > 2035).fillna(False).astype(int)
for col in ("contract_amount", "total_contract_amount", "reserve_price"):
    c[col] = to_int(c[col])
c["biz_type"] = c["biz_type_name"]
assert c["biz_type"].isin(["물품", "용역"]).all()
c["joint_contract_yn"] = (c["joint_contract_yn"] == "공동계약").astype("Int64")
c["sido_code"] = sido_code(c["vendor_address"])
c["raw_row_id"] = c["row_id"]
c["cleaned_at"] = NOW; c["cleaned_by"] = USER
print("중복 처리 후:", len(c), "| 최종 차수 행:", int(c["is_latest_seq"].sum()), "| 계약번호 고유:", c["contract_no"].nunique())
print("기간 이상치:", int(c["period_anomaly_flag"].sum()), "| 총계약금액 NULL:", int(c["total_contract_amount"].isna().sum()),
      "| 예정가격 NULL:", int(c["reserve_price"].isna().sum()), "| sido_code NULL:", int(c["sido_code"].isna().sum()))
print("sido 미매핑 토큰:", c.loc[c["sido_code"].isna(), "vendor_address"].astype(str).str.split(" ").str[0].value_counts().to_dict())   # 기대 {'**': 1, '1': 1} (용산구 2행은 테스트 업체로 제외됨)
gj = c["vendor_address"].astype(str).str.strip().str.split().str[0].isin(["광주", "광주시"])
print("광주 토큰:", int(gj.sum()), "행 → 29:", int((gj & c["sido_code"].eq("29")).sum()), "| 41:", int((gj & c["sido_code"].eq("41")).sum()),
      "| AMBIGUOUS_GWANGJU(NULL):", int((gj & c["sido_code"].isna()).sum()))   # 기대 1 → 29:1
print("연도별:", pd.to_datetime(c["contract_date"]).dt.year.value_counts().sort_index().to_dict())

원본 전체: 43112
정규화 후 키 고유: 43111 | 충돌 키: 1 행: 2
  2024UMM1504-01: row_id [15466, 15467] | 다른 열: 없음(원본 완전 중복)


테스트 업체 제외: 6 행 ['2024SFC0165', '2025LMH0104', '2025SFC0086', '2025LMZ0052', '2025LMZ0053', '2025LMZ0056']


중복 처리 후: 43105 | 최종 차수 행: 37602 | 계약번호 고유: 37602
기간 이상치: 1 | 총계약금액 NULL: 1 | 예정가격 NULL: 36 | sido_code NULL: 2
sido 미매핑 토큰: {'**': 1, '1': 1}


광주 토큰: 1 행 → 29: 1 | 41: 0 | AMBIGUOUS_GWANGJU(NULL): 0
연도별: {2024: 12304, 2025: 30801}


## 2. 입찰공고 → `clean_dapa_bid_notice`

- 키 = `ref_notice_no` × `ref_notice_seq_norm`(raw 고유 10,842 = 전체). `bid_notice_no`+`bid_notice_seq_norm`은 비유일(10,486)이라 입찰결과 연결용 KEY만. 차수는 한 자리(1~5)라 zfill만 한다.
- **열 밀림 2행**(`LCF0223` 1·2차: `bid_notice_date` NULL, `contract_method_name`='Y', `joint_contract_yn`에 날짜) → `clean_excluded_row` COL_SHIFT. 나머지 10,840행은 Y/N·날짜·금액 형식 위반 0.
- `bid_notice_status` = 상태명에서 `공고` 접미 제거(긴급/취소/정상/재공고/정정/연기). 면허제한 8열 → 개수 + `|` 결합. 담당자명 2열·시각 열은 넣지 않는다.

In [3]:
rn = raw("raw_dapa_bid_notice", ["bid_notice_no", "bid_notice_seq", "ref_notice_no", "ref_notice_seq", "g2b_notice_yn", "bid_notice_name", "bid_notice_status_name", "bid_notice_date", "biz_type_name", "joint_contract_yn", "joint_supply_method_name", "e_bid_yn", "contract_form_name", "contract_method_name", "award_method_name", "notice_org_name", "notice_org_code", "notice_org_dept_name", "demand_org_name", "demand_org_code", "demand_org_dept_name", "briefing_yn", "briefing_date", "briefing_place", "qualification_deadline_date", "bid_deadline_date", "opening_date", "opening_place", "budget_amount", "allocated_budget_amount", "region_limit_yn", "eligible_region_name", "license_limit_group1", "license_limit_group2", "license_limit_group3", "license_limit_group4", "license_limit_group5", "license_limit_group6", "license_limit_group7", "license_limit_group8", "bid_notice_url"])
print("원본 전체:", len(rn))
shift = rn[~rn["bid_notice_date"].astype(str).str.fullmatch(DATE_RE)]
print("열 밀림(공고일자 비정상):", len(shift), shift[["row_id", "bid_notice_no", "bid_notice_seq", "bid_notice_date", "contract_method_name", "joint_contract_yn"]].values.tolist())
for r in shift.itertuples():
    excluded.append(("raw_dapa_bid_notice", int(r.row_id), "COL_SHIFT",
                     f"{r.bid_notice_no}-{r.bid_notice_seq}: 입찰공고일자 NULL, 계약체결방법명='{r.contract_method_name}', 공동계약여부='{r.joint_contract_yn}'(날짜) — 열이 한 칸 밀린 원본"))
n = rn.drop(shift.index).copy()
n["ref_notice_seq_norm"] = n["ref_notice_seq"].astype(str).str.zfill(2)
n["bid_notice_seq_norm"] = n["bid_notice_seq"].astype(str).str.zfill(2)
assert not n.duplicated(["ref_notice_no", "ref_notice_seq_norm"]).any(), "참조공고번호+차수 비유일"
n["bid_notice_status"] = n["bid_notice_status_name"].str.replace("공고$", "", regex=True).replace({"재": "재공고"})
n["bid_notice_date"] = to_date(n["bid_notice_date"])
for col in ("briefing_date", "qualification_deadline_date", "bid_deadline_date", "opening_date"):
    n[col] = to_date(n[col])
for col in ("g2b_notice_yn", "joint_contract_yn", "e_bid_yn", "briefing_yn", "region_limit_yn"):
    n[col] = yn(n[col])
n["biz_type"] = n["biz_type_name"]
n["budget_amount_krw"] = to_int(n["budget_amount"])
n["allocated_budget_amount_krw"] = to_int(n["allocated_budget_amount"])
lic = n[[f"license_limit_group{i}" for i in range(1, 9)]].fillna("").astype(str).apply(lambda s: s.str.strip())
n["license_limit_group_count"] = (lic != "").sum(axis=1).astype(int)
n["license_limit_groups"] = lic.apply(lambda r: " | ".join(v for v in r if v) or None, axis=1)
n["raw_row_id"] = n["row_id"]; n["cleaned_at"] = NOW; n["cleaned_by"] = USER
print("중복 처리 후(열 밀림 제외):", len(n), "| 공고번호+차수 고유:", n[["bid_notice_no", "bid_notice_seq_norm"]].drop_duplicates().shape[0])
print("상태:", n["bid_notice_status"].value_counts().to_dict())
print("면허제한 그룹 수 분포:", n["license_limit_group_count"].value_counts().sort_index().to_dict(), "| 예산 NULL:", int(n["budget_amount_krw"].isna().sum()))

원본 전체: 10842
열 밀림(공고일자 비정상): 2 [[10841, 'LCF0223', '2', None, 'Y', '2025-05-28'], [10842, 'LCF0223', '1', None, 'Y', '2025-05-15']]


중복 처리 후(열 밀림 제외): 10840 | 공고번호+차수 고유: 10484
상태: {'긴급': 5677, '정상': 2433, '재공고': 1409, '취소': 981, '정정': 324, '연기': 16}
면허제한 그룹 수 분포: {0: 7336, 1: 2170, 2: 424, 3: 101, 4: 140, 5: 569, 6: 14, 7: 62, 8: 24} | 예산 NULL: 596


## 3. 입찰결과 → `clean_dapa_bid_result`

- 키 = `bid_notice_no` × `bid_notice_seq_norm` × `result_seq`(같은 키 안 `row_id` 순 1..n). 199키 403행 중복은 삭제하지 않고 `key_row_count`·`dup_kind`로 보존, `is_key_representative` = `result_seq`=1(키당 1행, `v_bid_result_summary` 키 수 7,201 대조).
- `dup_kind`(실측): 개찰결과 값이 다르면 `결과 상이`(73), 결과 같고 낙찰자 여럿이면 `복수 낙찰`(108), 결과·낙찰자 같으면 `동일 결과 반복`(18, 재개찰 16 + 같은 날 기초금액 상이 2). `결과 상이` 키의 대표 행이 `row_id` 최소인 것은 **잠정 규칙**(최신 결과를 대표로 볼지 팀 확인).
- **열 밀림 2행**(`LCF0223` 1·2차: 개찰결과에 날짜, 적격심사여부 '제한경쟁') → COL_SHIFT. 남은 7,403행은 `유찰 1,746 / 개찰완료 5,275 / 순위확정 382`.
- 금액: 소수(예정가격 346·기초금액 24행)와 지수 표기는 반올림해 BIGINT. `is_awarded` = 최종낙찰금액이 숫자(개찰완료 중 185행은 금액 없음 → 0). `winner_biz_reg_no`는 12자 패턴만(유찰·순위확정은 구조적 NULL).
- `notice_link_status`: §2의 clean 공고에서 (`bid_notice_no`, `bid_notice_seq_norm`) 건수 0/1/2+ → 미연결/1:1/다중. 행 단위 조인은 하지 않는다(다중 303키에서 7,545행으로 증식).

In [4]:
rr = raw("raw_dapa_bid_result", ["bid_notice_no", "bid_notice_seq", "bid_notice_name", "biz_type_name", "contract_form_name", "contract_method_name", "award_method_name", "qualification_review_yn", "notice_org_name", "notice_org_code", "demand_org_name", "demand_org_code", "award_lower_limit_rate", "reserve_price", "base_amount", "estimated_price", "opening_date", "opening_result_name", "final_award_amount", "final_award_rate", "final_award_date", "winner_name", "winner_biz_reg_no", "winner_address"])
print("원본 전체:", len(rr))
shift = rr[~rr["opening_result_name"].isin(["유찰", "개찰완료", "순위확정"])]
print("열 밀림(개찰결과 비정상):", len(shift), shift[["row_id", "bid_notice_no", "bid_notice_seq", "opening_result_name", "qualification_review_yn"]].values.tolist())
for r in shift.itertuples():
    excluded.append(("raw_dapa_bid_result", int(r.row_id), "COL_SHIFT",
                     f"{r.bid_notice_no}-{r.bid_notice_seq}: 개찰결과구분명='{r.opening_result_name}'(날짜), 적격심사여부='{r.qualification_review_yn}' — 열이 밀린 원본(LCF0223)"))
b = rr.drop(shift.index).copy()
b["bid_notice_seq_norm"] = b["bid_notice_seq"].astype(str).str.zfill(2)
key = ["bid_notice_no", "bid_notice_seq_norm"]
b = b.sort_values("row_id")
b["result_seq"] = b.groupby(key)["row_id"].rank(method="first").astype(int)
b["key_row_count"] = b.groupby(key)["row_id"].transform("size").astype(int)
b["winner_biz_reg_no"] = b["winner_biz_reg_no"].where(b["winner_biz_reg_no"].astype(str).str.fullmatch(r"\d{3}-\d{2}-\d{5}"), None)
n_result = b.groupby(key)["opening_result_name"].transform("nunique")
n_winner = b.groupby(key)["winner_biz_reg_no"].transform(lambda s: s.fillna("").nunique())
b["dup_kind"] = np.select([b["key_row_count"] == 1, n_result > 1, n_winner > 1], ["단일", "결과 상이", "복수 낙찰"], default="동일 결과 반복")
b["is_key_representative"] = (b["result_seq"] == 1).astype(int)
b["biz_type"] = b["biz_type_name"]
b["qualification_review_yn"] = yn(b["qualification_review_yn"])
b["award_lower_limit_rate"] = to_dec(b["award_lower_limit_rate"])
for src, dst in (("reserve_price", "reserve_price_krw"), ("base_amount", "base_amount_krw"), ("estimated_price", "estimated_price_krw"), ("final_award_amount", "final_award_amount_krw")):
    b[dst] = to_int(b[src])
b["final_award_rate"] = to_dec(b["final_award_rate"])
b["opening_date"] = to_date(b["opening_date"]); assert b["opening_date"].notna().all()
b["final_award_date"] = to_date(b["final_award_date"])
b["opening_result"] = b["opening_result_name"]
b["is_awarded"] = b["final_award_amount_krw"].notna().astype(int)
b["winner_sido_code"] = sido_code(b["winner_address"])
link_n = n.groupby(key).size()
cnt = pd.MultiIndex.from_frame(b[key]).map(lambda k: link_n.get(k, 0))
b["notice_link_status"] = np.select([cnt == 0, cnt == 1], ["미연결", "1:1"], default="다중")
b["raw_row_id"] = b["row_id"]; b["cleaned_at"] = NOW; b["cleaned_by"] = USER
keys = b.drop_duplicates(key)
print("중복 처리 후(열 밀림 제외):", len(b), "| 키:", len(keys), "| 대표 행:", int(b["is_key_representative"].sum()))
print("dup_kind(키 기준):", keys["dup_kind"].value_counts().to_dict())
print("개찰결과:", b["opening_result"].value_counts().to_dict(), "| is_awarded:", int(b["is_awarded"].sum()),
      "| 개찰완료인데 낙찰금액 없음:", int(((b["opening_result"] == "개찰완료") & (b["is_awarded"] == 0)).sum()))
print("notice_link_status(키 기준):", keys["notice_link_status"].value_counts().to_dict(), "| 행 기준 매칭:", int((b["notice_link_status"] != "미연결").sum()))
print("낙찰 사업자번호 12자:", int(b["winner_biz_reg_no"].notna().sum()), "| 낙찰 주소 sido NULL(사업자번호 있는 행):", int((b["winner_biz_reg_no"].notna() & b["winner_sido_code"].isna()).sum()))

원본 전체: 7405
열 밀림(개찰결과 비정상): 2 [[3698, 'LCF0223', '1', '2025-05-21', '제한경쟁'], [4135, 'LCF0223', '2', '2025-06-05', '제한경쟁']]


중복 처리 후(열 밀림 제외): 7403 | 키: 7199 | 대표 행: 7199
dup_kind(키 기준): {'단일': 7000, '복수 낙찰': 108, '결과 상이': 73, '동일 결과 반복': 18}
개찰결과: {'개찰완료': 5275, '유찰': 1746, '순위확정': 382} | is_awarded: 5275 | 개찰완료인데 낙찰금액 없음: 0
notice_link_status(키 기준): {'1:1': 6567, '미연결': 329, '다중': 303} | 행 기준 매칭: 7070
낙찰 사업자번호 12자: 5275 | 낙찰 주소 sido NULL(사업자번호 있는 행): 0


## 4. 국내 조달계획 → `clean_dapa_domestic_plan` · 5. 군별 계약집행 → `clean_dapa_contract_exec_by_service`

- 조달계획은 raw 1:1(PK `raw_row_id`). `decision_no`는 중복 824(고유 35,035)라 UNIQUE 불가 → `decision_row_count`로 보존. `plan_month`는 원본이 `YYYY-MM-DD`(전부 1일인지 출력). `plan_year`=2024는 4,545행 불완전 연도(`is_partial_year=1`). `budget_krw`는 지수 표기 11행을 정수로, 미기재 4,965행은 NULL(0 아님). `exec_type`은 TRIM(후행 공백 2값, 7종 유지). `is_contracted` = `진행상태`='계약완료'.
- 군별 계약집행은 연도×군 40행, 억원 단위 그대로.

In [5]:
rp = raw("raw_dapa_domestic_plan", ["plan_month", "decision_no", "rep_item_name", "exec_type", "contract_method", "exec_agency", "budget_amount", "bid_method", "progress_status"])
print("원본 전체:", len(rp))
p = rp.copy()
p["decision_row_count"] = p.groupby("decision_no")["row_id"].transform("size").astype(int)
p["plan_month"] = to_date(p["plan_month"]); assert p["plan_month"].notna().all()
pm = pd.to_datetime(p["plan_month"])
print("plan_month 일(day) 분포:", pm.dt.day.value_counts().to_dict())
p["plan_year"] = pm.dt.year.astype(str)
p["is_partial_year"] = (p["plan_year"] == "2024").astype(int)
p["exec_type"] = p["exec_type"].astype(str).str.strip()
p["budget_krw"] = to_int(p["budget_amount"])
p["is_contracted"] = (p["progress_status"] == "계약완료").astype(int)
p["raw_row_id"] = p["row_id"]; p["cleaned_at"] = NOW; p["cleaned_by"] = USER
print("판단번호 고유:", p["decision_no"].nunique(), "| 중복 행(decision_row_count>1):", int((p["decision_row_count"] > 1).sum()))
print("연도별 행·예산(억원)·계약완료:", p.groupby("plan_year").agg(n=("row_id", "size"), budget_100m=("budget_krw", lambda s: round(s.sum() / 1e8)), contracted=("is_contracted", "sum")).to_dict("index"))
print("exec_type:", p["exec_type"].value_counts().to_dict(), "| budget NULL:", int(p["budget_krw"].isna().sum()))

re_ = raw("raw_dapa_contract_exec_by_service", ["year", "service_branch", "contract_amount_100m_krw"])
e = re_.copy()
e["year"] = e["year"].astype(int)
e["contract_amount_100m_krw"] = to_dec(e["contract_amount_100m_krw"])
assert e["service_branch"].isin(["육군", "해군", "공군", "국직"]).all() and e["contract_amount_100m_krw"].notna().all()
assert not e.duplicated(["year", "service_branch"]).any()
e["raw_row_id"] = e["row_id"]; e["cleaned_at"] = NOW; e["cleaned_by"] = USER
print("군별 계약집행:", len(e), "행 |", e["year"].min(), "~", e["year"].max(), "|", e["service_branch"].value_counts().to_dict())

원본 전체: 35859
plan_month 일(day) 분포: {1: 35859}


판단번호 고유: 35035 | 중복 행(decision_row_count>1): 1623
연도별 행·예산(억원)·계약완료: {'2024': {'n': 4545, 'budget_100m': 10754, 'contracted': 3431}, '2025': {'n': 31314, 'budget_100m': 85577, 'contracted': 24013}}
exec_type: {'구매': 31361, '제조': 2432, '제조/구매': 1669, '기타': 173, '공사': 111, '리스': 97, '공급': 16} | budget NULL: 4965
군별 계약집행: 40 행 | 2015 ~ 2024 | {'육군': 10, '해군': 10, '공군': 10, '국직': 10}


## 6. 업체 마스터 `clean_company` · 이름 연결 `clean_company_name_link`

- `clean_company`: 계약정보 `vendor_biz_reg_no`(고유 14,723) ∪ 낙찰업체 `winner_biz_reg_no`(고유 3,210, 계약정보에 없는 116) = 14,839. 사업자번호당 업체명 변이 0(실측)이라 `name_raw`는 유일값. `address`는 계약정보의 최신 계약 행 주소, 계약정보에 없으면 낙찰 최신 행 주소. `first_seen_source` = 계약정보에 있으면 `contract`, 아니면 `bid_result`.
- `clean_company_name_link`: 사업자번호 없는 두 출처의 업체명을 `name_norm`으로 `clean_company.name_norm`과 대조. `defense_company` 84건(실측 exact 49 / multi 2 / none 33), `localized_item` 고유 원문 407건(정규화 후 403, 128 일치). `match_type`=exact일 때만 `biz_reg_no`를 채운다. **연결률·다중·미연결 수를 화면에 병기한 뒤에만 사용**(`data-cleaning-rules.md` §3).

In [6]:
cc = c[["vendor_biz_reg_no", "vendor_name", "vendor_address", "sido_code", "contract_date", "row_id"]].rename(
    columns={"vendor_biz_reg_no": "biz_reg_no", "vendor_name": "name_raw", "vendor_address": "address", "contract_date": "d"})
cc["first_seen_source"] = "contract"
bb = b.loc[b["winner_biz_reg_no"].notna(), ["winner_biz_reg_no", "winner_name", "winner_address", "winner_sido_code", "opening_date", "row_id"]].rename(
    columns={"winner_biz_reg_no": "biz_reg_no", "winner_name": "name_raw", "winner_address": "address", "winner_sido_code": "sido_code", "opening_date": "d"})
bb["first_seen_source"] = "bid_result"
u = pd.concat([cc, bb], ignore_index=True)
u["src_rank"] = (u["first_seen_source"] == "bid_result").astype(int)   # 계약정보 우선
u = u.sort_values(["src_rank", "d", "row_id"], ascending=[True, False, False])
names = u.groupby("biz_reg_no")["name_raw"].nunique()
print("사업자번호 고유:", u["biz_reg_no"].nunique(), "| 업체명 변이 있는 사업자번호:", int((names > 1).sum()))
comp = u.drop_duplicates("biz_reg_no").copy()      # 출처 우선 → 최신 날짜 행
comp["name_norm"] = name_norm(comp["name_raw"])
print("clean_company 후보:", len(comp), "|", comp["first_seen_source"].value_counts().to_dict(), "| name_norm 빈값:", int((comp["name_norm"] == "").sum()))

norm_map = comp.groupby("name_norm")["biz_reg_no"].agg(["nunique", "min"])
def link(source, raw_names):
    df = pd.DataFrame({"name_raw": pd.Series(raw_names).dropna().astype(str).str.strip()})
    df = df[df["name_raw"] != ""].drop_duplicates("name_raw").copy()
    df["source"] = source
    df["name_norm"] = name_norm(df["name_raw"])
    hit = df["name_norm"].map(norm_map["nunique"]).fillna(0).astype(int)
    df["match_count"] = hit
    df["match_type"] = np.select([hit == 0, hit == 1], ["none", "exact"], default="multi")
    df["biz_reg_no"] = np.where(df["match_type"] == "exact", df["name_norm"].map(norm_map["min"]), None)
    df["note"] = np.where(df["match_type"] == "multi", "정규화명이 사업자번호 " + hit.astype(str) + "개에 대응 — 수동 확인 필요", None)
    return df
dc = raw("raw_dapa_defense_company", ["company_name"])
_li = raw("raw_dapa_localized_item")["contractor_name"].dropna().astype(str).str.strip()
li = pd.DataFrame({"contractor_name": _li[_li != ""].unique()})
lk = pd.concat([link("defense_company", dc["company_name"]), link("localized_item", li["contractor_name"])], ignore_index=True)
for s, grp in lk.groupby("source"):
    vc = grp["match_type"].value_counts().to_dict()
    print(f"{s}: {len(grp)}건 → {vc} | 연결률 {vc.get('exact', 0) / len(grp) * 100:.1f}%")
display(lk[(lk["source"] == "defense_company") & (lk["match_type"] != "exact")][["name_raw", "match_type", "match_count"]].reset_index(drop=True))

사업자번호 고유: 14836 | 업체명 변이 있는 사업자번호: 0
clean_company 후보: 14836 | {'contract': 14721, 'bid_result': 115} | name_norm 빈값: 0


defense_company: 84건 → {'exact': 49, 'none': 33, 'multi': 2} | 연결률 58.3%
localized_item: 407건 → {'none': 277, 'exact': 118, 'multi': 12} | 연결률 29.0%


,name_raw,match_type,match_count
0,HD현대중공업,none,0
1,HJ중공업,none,0
2,LIG넥스원,none,0
3,LS엠트론,none,0
4,SG생활안전,none,0
5,SG솔루션,none,0
6,SK오션플랜트,none,0
7,SNT다이내믹스,none,0
8,SNT모티브,none,0
9,STX엔진,none,0


## 7. 적재 (대상이 비어 있을 때만) + `clean_excluded_row` + `meta_load_log`

In [7]:
if count("clean_dapa_contract") > 0:
    print("clean_dapa_contract 비어 있지 않음 → 건너뜀")
else:
    log_raw_stage(cur, "raw_dapa_contract", raw("raw_dapa_contract"), USER)   # 원본 전체 단계(파일 파서 행 수)
    # 저분산 원본 속성 제외: contract_form_name·joint_contract_yn·demand_org_name — raw에만 보존
    cols = ["contract_no", "contract_seq_norm", "raw_row_id", "contract_name", "biz_type", "contract_method_name",
            "contract_date", "period_start", "period_end", "period_anomaly_flag", "contract_amount", "total_contract_amount",
            "reserve_price", "contract_org_name", "vendor_name", "vendor_biz_reg_no", "vendor_address", "sido_code",
            "contract_type", "is_latest_seq", "seq_conflict_flag", "conflict_raw_row_ids", "evidence", "cleaned_at", "cleaned_by"]
    n_ins = insert_df("clean_dapa_contract", c, cols)
    log_stage(cur, "dapa_contract", "clean_dapa_contract", "중복 처리 후", "계약번호×차수(0→00) 고유", n_ins,
              "키 충돌 1건(2024UMM1504-01, 원본 완전 중복 2행)은 대표 행 1개만 적재, 나머지는 clean_excluded_row KEY_CONFLICT",
              f"{NB_TAG} §1", USER)
    log_stage(cur, "dapa_contract", "clean_dapa_contract", "중복 처리 후", "계약 단위(is_latest_seq=1)", int(c["is_latest_seq"].sum()),
              "변경계약 차수를 계약번호당 최종 차수 1행으로 축약(이력은 clean에 보존)", f"{NB_TAG} §1", USER)
    conn.commit(); print("clean_dapa_contract 적재:", count("clean_dapa_contract"))

if count("clean_dapa_bid_notice") > 0:
    print("clean_dapa_bid_notice 비어 있지 않음 → 건너뜀")
else:
    log_raw_stage(cur, "raw_dapa_bid_notice", raw("raw_dapa_bid_notice"), USER)   # 원본 전체 단계(파일 파서 행 수)
    # 저분산 원본 속성 10열 제외: g2b_notice_yn·joint_contract_yn·joint_supply_method_name·e_bid_yn·
    #   briefing_yn·briefing_date·briefing_place·opening_place·region_limit_yn·eligible_region_name — raw에만 보존(열 밀림 검출·Y/N 변환 로직은 위에서 유지)
    cols = ["ref_notice_no", "ref_notice_seq_norm", "raw_row_id", "bid_notice_no", "bid_notice_seq_norm", "bid_notice_name", "bid_notice_status",
            "bid_notice_date", "biz_type", "contract_form_name",
            "contract_method_name", "award_method_name", "notice_org_name", "notice_org_code", "notice_org_dept_name", "demand_org_name",
            "demand_org_code", "demand_org_dept_name", "qualification_deadline_date",
            "bid_deadline_date", "opening_date", "budget_amount_krw", "allocated_budget_amount_krw",
            "license_limit_group_count", "license_limit_groups", "bid_notice_url", "cleaned_at", "cleaned_by"]
    n_ins = insert_df("clean_dapa_bid_notice", n, cols)
    log_stage(cur, "dapa_bid_notice", "clean_dapa_bid_notice", "중복 처리 후", "참조공고번호+차수 고유(열 밀림 2행 제외)", n_ins,
              "열 밀림 원본 2행(LCF0223 1·2차)은 clean_excluded_row COL_SHIFT", f"{NB_TAG} §2", USER)
    conn.commit(); print("clean_dapa_bid_notice 적재:", count("clean_dapa_bid_notice"))

if count("clean_dapa_bid_result") > 0:
    print("clean_dapa_bid_result 비어 있지 않음 → 건너뜀")
else:
    log_raw_stage(cur, "raw_dapa_bid_result", raw("raw_dapa_bid_result"), USER)   # 원본 전체 단계(파일 파서 행 수)
    cols = ["bid_notice_no", "bid_notice_seq_norm", "result_seq", "raw_row_id", "key_row_count", "dup_kind", "is_key_representative",
            "bid_notice_name", "biz_type", "contract_form_name", "contract_method_name", "award_method_name", "qualification_review_yn",
            "notice_org_name", "notice_org_code", "demand_org_name", "demand_org_code", "award_lower_limit_rate", "reserve_price_krw",
            "base_amount_krw", "estimated_price_krw", "opening_date", "opening_result", "is_awarded", "final_award_amount_krw", "final_award_rate",
            "final_award_date", "winner_name", "winner_biz_reg_no", "winner_address", "winner_sido_code", "notice_link_status", "cleaned_at", "cleaned_by"]
    n_ins = insert_df("clean_dapa_bid_result", b, cols)
    kd = keys["dup_kind"].value_counts().to_dict()
    log_stage(cur, "dapa_bid_result", "clean_dapa_bid_result", "중복 처리 후", "공고번호+차수+result_seq(열 밀림 2행 제외, 중복 키 행 보존)", n_ins,
              f"열 밀림 2행 COL_SHIFT. 중복 199키는 삭제하지 않고 dup_kind로 구분: {kd}", f"{NB_TAG} §3", USER)
    log_stage(cur, "dapa_bid_result", "clean_dapa_bid_result", "중복 처리 후", "키 단위(is_key_representative=1)", int(b["is_key_representative"].sum()),
              "키당 대표 1행(row_id 최소, 잠정 규칙)", f"{NB_TAG} §3", USER)
    conn.commit(); print("clean_dapa_bid_result 적재:", count("clean_dapa_bid_result"))

if count("clean_dapa_domestic_plan") > 0:
    print("clean_dapa_domestic_plan 비어 있지 않음 → 건너뜀")
else:
    log_raw_stage(cur, "raw_dapa_domestic_plan", raw("raw_dapa_domestic_plan"), USER)   # 원본 전체 단계(파일 파서 행 수)
    cols = ["raw_row_id", "decision_no", "decision_row_count", "plan_month", "plan_year", "is_partial_year", "rep_item_name", "exec_type",
            "contract_method", "exec_agency", "budget_krw", "progress_status", "is_contracted", "cleaned_at", "cleaned_by"]  # bid_method 제외(98.4% 총액제, raw 보존)
    n_ins = insert_df("clean_dapa_domestic_plan", p, cols)
    log_stage(cur, "dapa_domestic_plan", "clean_dapa_domestic_plan", "중복 처리 후", "raw 1:1(판단번호 중복 824는 decision_row_count로 보존)", n_ins,
              f"제외 0. 예산 미기재 {int(p['budget_krw'].isna().sum())}행은 NULL(0 아님)", f"{NB_TAG} §4", USER)
    conn.commit(); print("clean_dapa_domestic_plan 적재:", count("clean_dapa_domestic_plan"))

if count("clean_dapa_contract_exec_by_service") > 0:
    print("clean_dapa_contract_exec_by_service 비어 있지 않음 → 건너뜀")
else:
    log_raw_stage(cur, "raw_dapa_contract_exec_by_service", raw("raw_dapa_contract_exec_by_service"), USER)   # 원본 전체 단계(파일 파서 행 수)
    n_ins = insert_df("clean_dapa_contract_exec_by_service", e, ["year", "service_branch", "contract_amount_100m_krw", "raw_row_id", "cleaned_at", "cleaned_by"])
    log_stage(cur, "dapa_contract_exec_by_service", "clean_dapa_contract_exec_by_service", "중복 처리 후", "연도×군", n_ins, "제외 0", f"{NB_TAG} §5", USER)
    conn.commit(); print("clean_dapa_contract_exec_by_service 적재:", count("clean_dapa_contract_exec_by_service"))

if count("clean_company") > 0:
    print("clean_company 비어 있지 않음 → 건너뜀")
else:
    n_ins = insert_df("clean_company", comp, ["biz_reg_no", "name_norm", "name_raw", "address", "sido_code", "first_seen_source"])
    log_stage(cur, "dapa_contract", "clean_company", "중복 처리 후", "사업자번호 고유(계약정보 ∪ 낙찰업체)", n_ins,
              f"계약정보 {int((comp['first_seen_source']=='contract').sum())} + 낙찰업체에만 {int((comp['first_seen_source']=='bid_result').sum())}. 업체명 변이 0", f"{NB_TAG} §6", USER)
    conn.commit(); print("clean_company 적재:", count("clean_company"))

if count("clean_company_name_link") > 0:
    print("clean_company_name_link 비어 있지 않음 → 건너뜀")
else:
    n_ins = insert_df("clean_company_name_link", lk, ["source", "name_raw", "name_norm", "biz_reg_no", "match_type", "match_count", "note"])
    for s, dk in (("defense_company", "dapa_defense_company"), ("localized_item", "dapa_localized_item")):
        grp = lk[lk["source"] == s]; vc = grp["match_type"].value_counts().to_dict()
        log_stage(cur, dk, "clean_company_name_link", "관련 후보", f"{s} 업체명 → clean_company name_norm 일치(exact)", int(vc.get("exact", 0)),
                  f"전체 {len(grp)} 중 multi {vc.get('multi', 0)} · none {vc.get('none', 0)}. name_norm 잠정 규칙(법인격 5종·공백 제거). 사업자번호 없는 매칭은 후보", f"{NB_TAG} §6", USER)
    conn.commit(); print("clean_company_name_link 적재:", count("clean_company_name_link"))

ex = pd.DataFrame(excluded, columns=["table_name", "raw_row_id", "reason_code", "note"])
cur.execute("SELECT table_name, raw_row_id FROM clean_excluded_row")
have = set(cur.fetchall())
ex = ex[~ex.apply(lambda r: (r["table_name"], r["raw_row_id"]) in have, axis=1)]
if len(ex):
    ex["excluded_by"] = USER
    insert_df("clean_excluded_row", ex, ["table_name", "raw_row_id", "reason_code", "note", "excluded_by"])
    conn.commit()
print("clean_excluded_row 추가:", len(ex), "| 현재:", count("clean_excluded_row"))

if count("clean_dapa_defense_company") > 0:
    print("clean_dapa_defense_company 비어 있지 않음 → 건너뜀")
else:
    # v_defense_company_sector 원천. 84행 1:1, seq_no PK, 지정일자 DATE
    dcf = raw("raw_dapa_defense_company")
    dcf["seq_no"] = to_int(dcf["seq_no"]); dcf["designated_date"] = to_date(dcf["designated_date"])
    dcf["raw_row_id"] = dcf["row_id"]; dcf["cleaned_at"] = NOW; dcf["cleaned_by"] = USER
    n_ins = insert_df("clean_dapa_defense_company", dcf, ["seq_no", "company_name", "sector", "designated_date", "note", "raw_row_id", "cleaned_at", "cleaned_by"])
    log_raw_stage(cur, "raw_dapa_defense_company", dcf, USER)
    log_stage(cur, "dapa_defense_company", "clean_dapa_defense_company", "중복 처리 후", "지정 업체 1행", n_ins,
              "제외 0(84행 1:1). 분야 공란은 뷰에서 「미기재」", f"{NB_TAG} §6", USER)
    conn.commit(); print("clean_dapa_defense_company 적재:", count("clean_dapa_defense_company"))


clean_dapa_contract 비어 있지 않음 → 건너뜀
clean_dapa_bid_notice 비어 있지 않음 → 건너뜀
clean_dapa_bid_result 비어 있지 않음 → 건너뜀
clean_dapa_domestic_plan 비어 있지 않음 → 건너뜀
clean_dapa_contract_exec_by_service 비어 있지 않음 → 건너뜀
clean_company 비어 있지 않음 → 건너뜀


clean_company_name_link 비어 있지 않음 → 건너뜀
clean_excluded_row 추가: 0 | 현재: 17
clean_dapa_defense_company 비어 있지 않음 → 건너뜀


## 8. 적재 후 검증 — 검산 `raw = clean + excluded`, 키 규칙, 뷰가 값을 내는지

In [8]:
_pairs = [("raw_dapa_contract", "clean_dapa_contract"), ("raw_dapa_bid_notice", "clean_dapa_bid_notice"), ("raw_dapa_bid_result", "clean_dapa_bid_result"), ("raw_dapa_domestic_plan", "clean_dapa_domestic_plan"), ("raw_dapa_contract_exec_by_service", "clean_dapa_contract_exec_by_service"), ("raw_dapa_defense_company", "clean_dapa_defense_company")]
# 원본 = 파일(read_raw), 정제·제외 = DB. raw = clean + excluded 검산
chk = pd.DataFrame({"tbl": [r for r, _ in _pairs], "raw_n": [len(raw(r)) for r, _ in _pairs], "clean_n": [count(c) for _, c in _pairs]})
_ex = q("SELECT table_name tbl, COUNT(*) excl_n FROM clean_excluded_row GROUP BY table_name")
chk = chk.merge(_ex, on="tbl", how="left").fillna({"excl_n": 0})
chk["excl_n"] = chk["excl_n"].astype(int); chk["gap"] = chk["raw_n"] - chk["clean_n"] - chk["excl_n"]
display(chk)
display(q("SELECT reason_code, table_name, COUNT(*) n FROM clean_excluded_row GROUP BY 1, 2"))
display(q("SELECT SUM(is_latest_seq) latest_rows, COUNT(DISTINCT contract_no) contracts, SUM(seq_conflict_flag) conflicts, SUM(period_anomaly_flag) period_anomaly, "
          "SUM(sido_code IS NULL) sido_null, SUM(class5='판단 보류') class5_default FROM clean_dapa_contract"))
display(q("SELECT contract_no, COUNT(*) n FROM clean_dapa_contract WHERE is_latest_seq=1 GROUP BY contract_no HAVING n<>1"))   # 0행 기대
display(q("SELECT dup_kind, COUNT(*) rows_, COUNT(DISTINCT bid_notice_no, bid_notice_seq_norm) keys_ FROM clean_dapa_bid_result GROUP BY 1"))
display(q("SELECT notice_link_status, COUNT(DISTINCT bid_notice_no, bid_notice_seq_norm) keys_, COUNT(*) rows_ FROM clean_dapa_bid_result GROUP BY 1"))
display(q("SELECT opening_result, COUNT(*) n, SUM(is_key_representative) rep FROM clean_dapa_bid_result GROUP BY 1"))
display(q("SELECT plan_year, COUNT(*) n, ROUND(SUM(budget_krw)/1e8) budget_100m, SUM(budget_krw IS NULL) budget_null, SUM(is_contracted) contracted FROM clean_dapa_domestic_plan GROUP BY 1"))
display(q("SELECT first_seen_source, COUNT(*) n, SUM(sido_code IS NULL) sido_null FROM clean_company GROUP BY 1"))
display(q("SELECT source, match_type, COUNT(*) n FROM clean_company_name_link GROUP BY 1, 2 ORDER BY 1, 2"))
display(q("SELECT COUNT(*) n_biz_result_rows, SUM(c.biz_reg_no IS NOT NULL) linked FROM clean_dapa_bid_result r LEFT JOIN clean_company c ON c.biz_reg_no = r.winner_biz_reg_no WHERE r.winner_biz_reg_no IS NOT NULL"))
display(q("SELECT * FROM v_contract_monthly ORDER BY 1, 2, 3 LIMIT 8"))
display(q("SELECT COUNT(*) n_rows FROM v_contract_monthly"))

,tbl,raw_n,clean_n,excl_n,gap
0,raw_dapa_contract,43112,43105,7,0
1,raw_dapa_bid_notice,10842,10840,2,0
2,raw_dapa_bid_result,7405,7403,2,0
3,raw_dapa_domestic_plan,35859,35859,0,0
4,raw_dapa_contract_exec_by_service,40,40,0,0
5,raw_dapa_defense_company,84,84,0,0


,reason_code,table_name,n
0,KEY_CONFLICT,raw_dapa_contract,1
1,COL_SHIFT,raw_dapa_bid_notice,2
2,COL_SHIFT,raw_dapa_bid_result,2
3,PLACEHOLDER,raw_dapa_contract,6
4,PLACEHOLDER,raw_dapa_overseas_contract,6


,latest_rows,contracts,conflicts,period_anomaly,sido_null,class5_default
0,37602,37602,1,1,2,43105


,contract_no,n


,dup_kind,rows_,keys_
0,단일,7000,7000
1,복수 낙찰,219,108
2,결과 상이,146,73
3,동일 결과 반복,38,18


...

,first_seen_source,n,sido_null
0,contract,14721,2
1,bid_result,115,0


,source,match_type,n
0,localized_item,exact,118
1,localized_item,multi,12
2,localized_item,none,277
3,defense_company,exact,49
4,defense_company,multi,2
5,defense_company,none,33


,n_biz_result_rows,linked
0,5275,5275


,yyyymm,biz_type,class5,contract_count,total_contract_amount,amount_missing_count
0,202411,물품,판단 보류,4698,766321223394,1
1,202411,용역,판단 보류,768,231652243531,0
2,202412,물품,판단 보류,5230,3471218628930,0
3,202412,용역,판단 보류,957,330633009211,0
4,202501,물품,판단 보류,654,236124138744,0
5,202501,용역,판단 보류,804,310561894070,0
6,202502,물품,판단 보류,833,173699785554,0
7,202502,용역,판단 보류,726,815052885358,0


,n_rows
0,28


## 남은 일 (이 노트북이 하지 않은 것)

- `class5`·전자/부품/방산 속성: 키워드 규칙표를 정한 뒤 UPDATE(`evidence`에 규칙 기록). 지금은 전부 `판단 보류`/`미확인`.
- `결과 상이` 중복 키의 대표 행(`row_id` 최소) 규칙 팀 확인 — 최신 개찰을 대표로 볼지.
- `clean_company_name_link` multi(방산업체 2건)·none 수동 검토. `name_norm` 규칙을 다른 정제본과 통일하면 필요할 때 재계산.
- `ref_sido_map` 미매핑: `충남대전시`는 30으로 매핑했다. 남은 NULL(`**` 1·`1` 1)은 판별 불가로 둔다(용산구 2행은 테스트 업체로 제외).
- 뷰 4개(`v_bid_*`·`v_domestic_plan_yearly`)는 원본 집계 그대로 두었다 — clean 기준 전환은 이번 범위 밖.